# 03. Recomendador Content-Based de recetas

Este notebook construye el primer modelo de recomendación del proyecto.

## Flujo

**CSV → TF-IDF → representación numérica → similitud coseno → Top-N**

La primera versión es un **baseline**: busca recetas parecidas por su contenido. Más adelante podremos añadir filtros, embeddings e interacciones de usuarios.

## 1. Librerías

Usaremos `pandas` para los datos y `scikit-learn` para TF-IDF y similitud coseno.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2. Localizar el dataset

El notebook estará en `notebooks/` y el CSV limpio en `data/processed/`.

La siguiente celda funciona tanto si Jupyter está usando como directorio de trabajo la raíz del proyecto como si está usando `notebooks/`.

In [2]:
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = PROJECT_DIR / "data" / "processed" / "recetas_unificadas_limpias.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el CSV: {DATA_PATH}")

print("Proyecto:", PROJECT_DIR)
print("CSV:", DATA_PATH)


Proyecto: /Users/i./Documents/DIPLOMADO/MODULO_5/chefWise
CSV: /Users/i./Documents/DIPLOMADO/MODULO_5/chefWise/data/processed/recetas_unificadas_limpias.csv


## 3. Cargar el CSV limpio

In [3]:
df = pd.read_csv(DATA_PATH)

print("Filas:", len(df))
print("Columnas:", len(df.columns))

df.head()


Filas: 28853
Columnas: 29


,recipe_id,recipe_id_source,source,name,url,ingredients,instructions,country,meal_type,app_category,...,rating_votes,num_comments,name_key,ingredients_key,has_ingredients,has_instructions,recipe_text,n_ingredients_approx,n_steps_approx,listed_time_min
0,1,3144,RecetasGratis,Burritos Veganos Faciles y Sabrosos,https://www.recetasgratis.net/receta-de-burrit...,1 lata de fruta de Jack o jackfruit | 1 cuchar...,1 Escurre la fruta de jack y lávala en un escu...,México,Entrante,comida,...,1.0,NaN,burritos veganos faciles y sabrosos,1 lata de fruta de jack o jackfruit 1 cucharad...,True,True,Burritos Veganos Faciles y Sabrosos | 1 lata d...,28.0,9.0,45.0
1,2,3129,RecetasGratis,Hallacas Venezolanas,https://www.recetasgratis.net/receta-de-hallac...,4 kilogramos de de carne | 4 kilogramos de coc...,1 Toda hallaca comienza por la preparación del...,Venezuela,Plato principal,comida,...,33.0,NaN,hallacas venezolanas,4 kilogramos de de carne 4 kilogramos de cochi...,True,True,Hallacas Venezolanas | 4 kilogramos de de carn...,28.0,9.0,240.0
2,3,2333,RecetasGratis,Pan con Chumpe Salvadoreno,https://www.recetasgratis.net/receta-de-pan-co...,1 Pavo (12-15 lb) | 2 Limones (jugo) | 4 cucha...,1 Lo primero que haremos será preparar el encu...,El Salvador,Plato principal,comida,...,51.0,NaN,pan con chumpe salvadoreno,1 pavo 12 15 lb 2 limones jugo 4 cucharadas so...,True,True,Pan con Chumpe Salvadoreno | 1 Pavo (12-15 lb)...,28.0,11.0,90.0
3,4,3631,RecetasGratis,Arroz Meloso Mar y Montana,https://www.recetasgratis.net/receta-de-arroz-...,250 gramos de Cabezas de gambas | 250 gramos d...,1 Para hacer que nuestra arroz mar y montaña t...,España,Plato principal,comida,...,2.0,NaN,arroz meloso mar y montana,250 gramos de cabezas de gambas 250 gramos de ...,True,True,Arroz Meloso Mar y Montana | 250 gramos de Cab...,26.0,17.0,90.0
4,5,5164,RecetasGratis,Lengua de Res Guisada,https://www.recetasgratis.net/receta-de-lengua...,1 pieza de Lengua de res | 750 gramos de Gramo...,"1 Se frota la lengua de res con limón. Luego, ...",México,Plato principal,comida,...,52.0,NaN,lengua de res guisada,1 pieza de lengua de res 750 gramos de gramos ...,True,True,Lengua de Res Guisada | 1 pieza de Lengua de r...,25.0,8.0,240.0


## 4. Revisar las columnas que usaremos

En este primer modelo usaremos principalmente `recipe_text`.

Esa columna ya fue creada durante la limpieza y reúne información textual útil de cada receta, como nombre, ingredientes y etiquetas.

No vamos a utilizar todavía `total_time_min`, `difficulty`, etc. dentro de TF-IDF. Esas variables numéricas/categóricas podrán funcionar después como **filtros** o como parte de un ranking más completo.

In [4]:
required = ["recipe_id", "name", "ingredients", "recipe_text"]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Faltan columnas: {missing}")

print("Nulos en recipe_text:", df["recipe_text"].isna().sum())
print("Textos vacíos:", (df["recipe_text"].fillna("").str.strip() == "").sum())
print()
print(df[["recipe_id", "name", "recipe_text"]].head(3).to_string(index=False))


Nulos en recipe_text: 0
Textos vacíos: 0

 recipe_id                                name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

## 5. Preparar el texto

TF-IDF recibe una colección de documentos.

Aquí:

- cada **receta** = un documento;
- `recipe_text` = el documento.

La computadora no puede comparar directamente frases como `"tacos de pollo"` y `"quesadillas de pollo"`, así que necesitamos convertirlas en vectores numéricos.

In [5]:
recipe_text = df["recipe_text"].fillna("").astype(str)

print("Número de recetas:", len(recipe_text))
print("Ejemplo:")
print(recipe_text.iloc[0][:500])


Número de recetas: 28853
Ejemplo:
Burritos Veganos Faciles y Sabrosos | 1 lata de fruta de Jack o jackfruit | 1 cucharada sopera de aceite | 1 cucharadita de ajo en polvo | 1 cucharadita de cebolla en polvo | 1 cucharadita de tomillo molido | 1 cucharadita de pimentón dulce | 1 pizca de pimienta cayena | 1 pizca de pimienta negra | 2 cucharadas soperas de zumo de limón | ½ cucharadita de canela molida | ½ cucharadita de nuez moscada molida | ½ cucharada sopera de azúcar moreno | 1½ tazas de agua (360 mililitros) | 1 taza de judí


## 6. TF-IDF

TF-IDF transforma cada receta en un vector.

Una palabra muy común en casi todas las recetas recibe menos peso. Una palabra que ayuda a distinguir unas recetas de otras recibe más peso.

Usamos `ngram_range=(1, 2)` para considerar:

- palabras individuales: `pollo`
- pares de palabras: `pollo asado`

**Importante:** no construiremos una matriz completa de similitud de 28 mil × 28 mil recetas. Eso produciría cientos de millones de comparaciones. Calcularemos la similitud contra todas las recetas únicamente cuando pidamos recomendaciones para una receta concreta.

In [6]:
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

tfidf_matrix = tfidf.fit_transform(recipe_text)

print("Filas (recetas):", tfidf_matrix.shape[0])
print("Columnas (términos):", tfidf_matrix.shape[1])
print("Tipo:", type(tfidf_matrix).__name__)


Filas (recetas): 28853
Columnas (términos): 63143
Tipo: csr_matrix


## 7. ¿Qué representa `tfidf_matrix`?

La matriz tiene:

```text
filas    = recetas
columnas = términos
```

Por tanto, cada fila es la representación numérica de una receta.

Ejemplo conceptual:

```text
Receta A → [0.0, 0.31, 0.82, 0.0, ...]
Receta B → [0.2, 0.29, 0.75, 0.0, ...]
Receta C → [0.0, 0.00, 0.00, 0.91, ...]
```

Las recetas A y B probablemente estarán más cerca entre sí que A y C.

## 8. Similitud coseno y Top-N

Ahora construimos el recomendador.

Para una receta seleccionada:

1. obtenemos su vector TF-IDF;
2. calculamos la similitud coseno contra todas las recetas;
3. ordenamos las similitudes;
4. eliminamos la propia receta;
5. devolvemos las `N` primeras.

Eso es nuestro **Top-N**.

In [7]:
def recomendar_similares(recipe_id, n=10):
    """Devuelve las N recetas más similares a una receta."""

    matches = df.index[df["recipe_id"] == recipe_id].tolist()

    if not matches:
        raise ValueError(f"No existe recipe_id={recipe_id}")

    idx = matches[0]

    # Vector TF-IDF de la receta seleccionada.
    query_vector = tfidf_matrix[idx]

    # Similitud con todas las recetas.
    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).ravel()

    # Índices ordenados de mayor a menor similitud.
    ranking = np.argsort(similarities)[::-1]

    # Quitamos la receta que usamos como consulta.
    ranking = [i for i in ranking if i != idx]

    top_indices = ranking[:n]

    result = df.iloc[top_indices][
        [
            "recipe_id",
            "name",
            "country",
            "difficulty",
            "total_time_min",
            "rating",
        ]
    ].copy()

    result["similarity"] = similarities[top_indices]

    return result.reset_index(drop=True)


## 9. Primera prueba

Vamos a probar con una receta mexicana real del dataset. No queremos evaluar todavía el modelo con una sola observación; esta prueba solamente verifica que el pipeline funcione.

In [8]:
mexicanas = df[
    df["country"].fillna("").str.lower().eq("méxico")
]

example = mexicanas[
    mexicanas["name"].str.contains("tacos", case=False, na=False)
].iloc[0]

print("recipe_id:", example["recipe_id"])
print("Receta:", example["name"])
print("Ingredientes:", example["ingredients"][:350])


recipe_id: 6
Receta: Tacos de Cerdo con Salsa Roja
Ingredientes: 20 unidades de Tortillas de maíz amarillo | 100 gramos de Lechuga picada | 200 gramos de Frijoles refritos | 700 gramos de Pierna de cerdo | 4 unidades de Tomate rojo | 2 unidades de Cebolla morada pequeña | 30 gramos de Cilantro | 1 unidad de Chile chipotle ahumado | 1 unidad de Limón | 50 mililitros de Aceite neutro | 3 dientes de Ajo | 5 gramos 


In [9]:
resultado = recomendar_similares(
    recipe_id=example["recipe_id"],
    n=10
)

resultado


,recipe_id,name,country,difficulty,total_time_min,rating,similarity
0,17,Burrito de Cerdo con Salsa Roja Ahumada,España,media,45.0,5.0,0.343331
1,4182,Tacos de Camaron y Pulpo,México,media,45.0,NaN,0.296952
2,534,Nachos Mexicanos con Guacamole,México,baja,30.0,4.7,0.282376
3,1390,Chuletas de Cerdo en Salsa de Pera,España,media,30.0,4.7,0.280356
4,3437,Albondigas Orientales con Salsa de Mani,Internacional,media,45.0,NaN,0.271891
5,16187,Mayonesa con Cebolla Ajo y Cilantro,Internacional,baja,15.0,4.9,0.262790
6,1379,Lomo en Salsa de Cebolla,España,media,30.0,4.8,0.259406
7,323,Tacos de Pollo Teriyaki,México,media,45.0,5.0,0.256132
8,511,Lomo Ancho a la Parrilla,Argentina,media,45.0,5.0,0.255403
9,1696,Dip de Remolacha y Cilantro,México,baja,45.0,5.0,0.254761


## 10. Interpretar el resultado

`similarity` es la similitud coseno.

- cerca de `1` → los vectores son muy parecidos;
- cerca de `0` → son poco parecidos.

Pero:

> **similitud de contenido no significa preferencia del usuario.**

Este modelo todavía no sabe qué persona prefiere qué recetas. Solo sabe comparar el contenido de las recetas.

Eso es exactamente lo que queremos para el primer baseline.

## 11. Consulta de texto

Como TF-IDF ya convierte texto en vectores, también podemos transformar una consulta escrita por el usuario.

Esto será útil posteriormente para preferencias como:

> `pollo jitomate chile`

Esto **no es el chatbot**. Es solo una forma de consultar el mismo espacio de recetas.

In [10]:
def recomendar_por_texto(texto, n=10):
    """Devuelve las N recetas más parecidas a una consulta escrita."""

    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("La consulta debe ser un texto no vacío.")

    query_vector = tfidf.transform([texto])

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).ravel()

    top_indices = np.argsort(similarities)[::-1][:n]

    result = df.iloc[top_indices][
        [
            "recipe_id",
            "name",
            "country",
            "difficulty",
            "total_time_min",
            "rating",
        ]
    ].copy()

    result["similarity"] = similarities[top_indices]

    return result.reset_index(drop=True)


In [11]:
consulta = "pollo jitomate chile"

recomendar_por_texto(consulta, n=10)


,recipe_id,name,country,difficulty,total_time_min,rating,similarity
0,17463,Chile Relleno de Queso Bañado en Salsa de Jito...,México,baja,30.0,4.500000,0.231063
1,6573,Bocoles Huastecos,México,baja,30.0,NaN,0.205672
2,16520,Caldo de Pollo con un Toque de Chile Chipotle,México,baja,35.0,2.500000,0.201811
3,12888,Papilla Irresistible de Corazón,México,baja,30.0,5.000000,0.200295
4,18469,Salchichas a la Mexicana,México,baja,10.0,4.666667,0.197803
5,16088,Arroz con Lentejas,México,baja,30.0,5.000000,0.197016
6,13589,Chiles Rellenos de Pollo en Salsa de Tomate,México,baja,30.0,5.000000,0.189799
7,2221,Frijoles Charros con Chile,México,baja,90.0,4.200000,0.175749
8,923,Burrito de Pollo a la Mexicana,México,baja,30.0,5.000000,0.175171
9,312,Enchiladas Suizas Rojas,México,baja,45.0,5.000000,0.173908


## 12. Guardar los artefactos del modelo

Una vez comprobado el baseline, guardamos el vectorizador y la matriz para no recalcularlos cada vez.

Estos archivos son **artefactos del modelo**, no datos procesados originales.

In [12]:
import joblib

MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.joblib")
joblib.dump(tfidf_matrix, MODELS_DIR / "recipe_tfidf_matrix.joblib")

print("Artefactos guardados en:", MODELS_DIR)


Artefactos guardados en: /Users/i./Documents/DIPLOMADO/MODULO_5/chefWise/models


# Resultado de esta etapa

Ya construimos:

```text
CSV limpio
    ↓
recipe_text
    ↓
TF-IDF
    ↓
vector de cada receta
    ↓
cosine similarity
    ↓
Top-N recomendaciones
```

## Lo que sigue

Antes de saltar al chatbot o a la base de datos, debemos **evaluar este baseline**.

La siguiente etapa será crear casos de prueba y métricas de ranking para responder:

> ¿Las recetas que recomienda realmente son relevantes?

Después podremos agregar filtros como `tiempo`, `dificultad`, `país` y preferencias explícitas, y comparar mejoras contra este baseline.